<a href="https://colab.research.google.com/github/vishalravi2k-amc/Intelligent-IDS-BCS501/blob/main/01_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
Real-Time Intrusion Detection System (IDS) using Synthetic Traffic Data
Project Code: VTU BCS501
Author: Vishal Ravi
"""

import time
from typing import Tuple
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


def generate_synthetic_data(
    num_samples: int = 5000, random_state: int = 42
) -> pd.DataFrame:
    """Generates synthetic network traffic metrics for normal and anomalous behaviors."""
    np.random.seed(random_state)

    normal_count = int(num_samples * 0.85)
    anomaly_count = num_samples - normal_count

    # Normal Network Traffic Profile
    normal_traffic = {
        'packet_length': np.random.normal(
            loc=500, scale=100, size=normal_count
        ),
        'duration': np.random.exponential(scale=2.0, size=normal_count),
        'failed_logins': np.random.poisson(lam=0.1, size=normal_count),
        'is_anomaly': np.zeros(normal_count),
    }

    # Anomalous / Attack Traffic Profile
    anomaly_traffic = {
        'packet_length': np.random.normal(
            loc=1400, scale=300, size=anomaly_count
        ),
        'duration': np.random.exponential(scale=0.1, size=anomaly_count),
        'failed_logins': np.random.poisson(lam=5.0, size=anomaly_count),
        'is_anomaly': np.ones(anomaly_count),
    }

    # Combine into unified dataset
    df_normal = pd.DataFrame(normal_traffic)
    df_anomaly = pd.DataFrame(anomaly_traffic)

    dataset = (
        pd.concat([df_normal, df_anomaly], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    return dataset


def train_anomaly_detector(
    df: pd.DataFrame,
) -> Tuple[RandomForestClassifier, pd.DataFrame, pd.Series]:
    """Trains a Random Forest classifier to identify network anomalies."""
    X = df.drop(columns=['is_anomaly'])
    y = df['is_anomaly']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42
    )
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    print("=== MODEL PERFORMANCE EVALUATION ===")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))
    print("\nClassification Report:")
    print(classification_report(y_test, predictions))

    return clf, X_test, y_test


def simulate_realtime_stream(
    model: RandomForestClassifier, X_test: pd.DataFrame, stream_count: int = 5
) -> None:
    """Simulates real-time network packet evaluation."""
    print("\n=== STARTING REAL-TIME ANOMALY DETECTION STREAM ===")
    for packet_id in range(1, stream_count + 1):
        sample = X_test.sample(1, random_state=packet_id)
        prediction = model.predict(sample)[0]

        status = (
            "⚠️ ALERT: Anomaly Detected"
            if prediction == 1
            else "✅ OK: Normal Traffic"
        )
        packet_data = sample.values[0].round(2)

        print(
            f"[Packet #{packet_id:02d}] Features (Length, Duration, FailedLogins): {packet_data} | Status: {status}"
        )
        time.sleep(0.4)

try
# --- EXECUTION FLOW ---
if __name__ == "__main__":
    # 1. Dataset Generation & Export
    data = generate_synthetic_data()
    data.to_csv("synthetic_ids_data.csv", index=False)
    print(f"Synthetic dataset saved with shape: {data.shape}\n")

    # 2. Model Training & Validation
    model, X_test, y_test = train_anomaly_detector(data)

    # 3. Real-Time Detection Simulation
    simulate_realtime_stream(model, X_test, stream_count=5)



Synthetic dataset saved with shape: (5000, 4)

=== MODEL PERFORMANCE EVALUATION ===

Confusion Matrix:
[[850   0]
 [  2 148]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       850
         1.0       1.00      0.99      0.99       150

    accuracy                           1.00      1000
   macro avg       1.00      0.99      1.00      1000
weighted avg       1.00      1.00      1.00      1000


=== STARTING REAL-TIME ANOMALY DETECTION STREAM ===
[Packet #01] Features (Length, Duration, FailedLogins): [334.51   1.01   0.  ] | Status: ✅ OK: Normal Traffic
[Packet #02] Features (Length, Duration, FailedLogins): [507.29   1.     0.  ] | Status: ✅ OK: Normal Traffic
[Packet #03] Features (Length, Duration, FailedLogins): [522.45   5.9    0.  ] | Status: ✅ OK: Normal Traffic
[Packet #04] Features (Length, Duration, FailedLogins): [429.04   2.96   0.  ] | Status: ✅ OK: Normal Traffic
[Packet #05] Features (Length,